In [4]:
!pip install pandas
!pip install langchain_community
!pip install langchain-tavily
!pip install seaborn
!piip install python-dotenv
!pip install openai
!pip install chromadb


zsh:1: command not found: piip


In [5]:
!pip install openai   
   

In [6]:
import openai
print("OpenAI installed successfully")

OpenAI installed successfully


In [7]:
import sys
print(sys.executable)

/opt/anaconda3/envs/tf-env/bin/python


In [8]:
import sys
!{sys.executable} -m pip install python-dotenv

In [9]:
import sys
!{sys.executable} -m pip install chromadb

In [10]:
import pandas as pd
import numpy as np
import json
import re
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
from dotenv import load_dotenv
import openai
import os

/opt/anaconda3/envs/tf-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
load_dotenv()

openai_api_key = os.getenv("OPEN_AI_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

In [12]:
df_qa = pd.read_csv("train.csv")
df_qa = df_qa.sample(500, random_state=0).reset_index(drop=True)

In [13]:
df_qa.head()

,qtype,Question,Answer
0,frequency,How many people are affected by X-linked chond...,The prevalence of X-linked chondrodysplasia pu...
1,treatment,What are the treatments for Kawasaki disease ?,These resources address the diagnosis or manag...
2,genetic changes,What are the genetic changes related to Ellis-...,Ellis-van Creveld syndrome can be caused by mu...
3,symptoms,What are the symptoms of Renal dysplasia-limb ...,What are the signs and symptoms of Renal dyspl...
4,information,What is (are) Fraser syndrome ?,Fraser syndrome is a rare disorder that affect...


In [14]:
df_qa['combined_text'] = (
    "Question: " + df_qa['Question'].astype(str) + '. ' +
    "Answer: " + df_qa['Answer'].astype(str) + '. '+
    "Type: " +df_qa['qtype'].astype(str)+'. '
)
df_qa.head()

,qtype,Question,Answer,combined_text
0,frequency,How many people are affected by X-linked chond...,The prevalence of X-linked chondrodysplasia pu...,Question: How many people are affected by X-li...
1,treatment,What are the treatments for Kawasaki disease ?,These resources address the diagnosis or manag...,Question: What are the treatments for Kawasaki...
2,genetic changes,What are the genetic changes related to Ellis-...,Ellis-van Creveld syndrome can be caused by mu...,Question: What are the genetic changes related...
3,symptoms,What are the symptoms of Renal dysplasia-limb ...,What are the signs and symptoms of Renal dyspl...,Question: What are the symptoms of Renal dyspl...
4,information,What is (are) Fraser syndrome ?,Fraser syndrome is a rare disorder that affect...,Question: What is (are) Fraser syndrome ?. Ans...


In [15]:
df_md = pd.read_csv('medical_device_manuals_dataset.csv')
df_md = df_md.sample(500, random_state=0).reset_index(drop=True)
df_md.head()

,Device_Name,Model_Number,Manufacturer,Manual_Version,Publication_Date,Device_Class,Regulatory_Approval_ID,Patient_Population,Indications_for_Use,Contraindications,Sterilization_Method,Number_of_Warnings,Number_of_Cautions,Device_Lifetime_Years,Device_Weight_kg,Max_Operating_Temperature_C
0,Electrosurgical Unit,Model 1606,Zimmer Biomet,2023-04-Z,2018-04-28,Class IIb,MDR-847127,All,Used for thermal therapy guidance in oncology ...,Contraindicated in presence of radio frequenci...,Hydrogen Peroxide Plasma,10,15,15.0,6.71,27.0
1,Dialysis Machine,X-6538,3M Healthcare,v2.8,2018-04-03,Class I,MDR-416480,Adult (>65),Used for post-operative chemotherapy managemen...,NaN,Gamma Irradiation,8,11,11.0,NaN,17.0
2,Ventilator,SON230,Sonova,Version 13,2022-02-24,Class III,BLA670706,Adult (>65),Indicated for real-time temperature assessment...,Do not use during general anesthesia administr...,Pre-Sterilized,18,28,11.0,149.42,25.0
3,Dialysis Machine,Max787,Boston Scientific,v4.7,2016-05-08,Class II,BLA131698,Neonatal,Intended for sterilization guidance during min...,"Not recommended during pregnancy, lactation, o...",NaN,10,15,15.0,20.96,20.0
4,Electrosurgical Unit,Plus691,Abbott,2020-03-Q,2015-01-30,Class III,H127393,Pediatric,Intended for life support evaluation in rehabi...,Contraindicated in patients with severe diabet...,Single-Use Sterile,13,17,8.0,7.75,31.0


In [16]:
df_md['combined_text'] = (
    "Device Name: " + df_md['Device_Name'].astype(str) + ". " +
    "Model: " + df_md['Model_Number'].astype(str) + ". " +
    "Manufacturer: " + df_md['Manufacturer'].astype(str) + ". " +
    "Indications: " + df_md['Indications_for_Use'].astype(str) + ". " +
    "Contraindications: " + df_md['Contraindications'].fillna('None').astype(str)
)
df_md.head()

,Device_Name,Model_Number,Manufacturer,Manual_Version,Publication_Date,Device_Class,Regulatory_Approval_ID,Patient_Population,Indications_for_Use,Contraindications,Sterilization_Method,Number_of_Warnings,Number_of_Cautions,Device_Lifetime_Years,Device_Weight_kg,Max_Operating_Temperature_C,combined_text
0,Electrosurgical Unit,Model 1606,Zimmer Biomet,2023-04-Z,2018-04-28,Class IIb,MDR-847127,All,Used for thermal therapy guidance in oncology ...,Contraindicated in presence of radio frequenci...,Hydrogen Peroxide Plasma,10,15,15.0,6.71,27.0,Device Name: Electrosurgical Unit. Model: Mode...
1,Dialysis Machine,X-6538,3M Healthcare,v2.8,2018-04-03,Class I,MDR-416480,Adult (>65),Used for post-operative chemotherapy managemen...,NaN,Gamma Irradiation,8,11,11.0,NaN,17.0,Device Name: Dialysis Machine. Model: X-6538. ...
2,Ventilator,SON230,Sonova,Version 13,2022-02-24,Class III,BLA670706,Adult (>65),Indicated for real-time temperature assessment...,Do not use during general anesthesia administr...,Pre-Sterilized,18,28,11.0,149.42,25.0,Device Name: Ventilator. Model: SON230. Manufa...
3,Dialysis Machine,Max787,Boston Scientific,v4.7,2016-05-08,Class II,BLA131698,Neonatal,Intended for sterilization guidance during min...,"Not recommended during pregnancy, lactation, o...",NaN,10,15,15.0,20.96,20.0,Device Name: Dialysis Machine. Model: Max787. ...
4,Electrosurgical Unit,Plus691,Abbott,2020-03-Q,2015-01-30,Class III,H127393,Pediatric,Intended for life support evaluation in rehabi...,Contraindicated in patients with severe diabet...,Single-Use Sterile,13,17,8.0,7.75,31.0,Device Name: Electrosurgical Unit. Model: Plus...


In [17]:
import chromadb
client = chromadb.PersistentClient(path= './chroma_db')

In [18]:
collection1 = client.get_or_create_collection(name='medical_qna')

In [19]:
df_qa = df_qa.fillna("")

# Convert everything to string (safe fix)
df_qa = df_qa.astype(str)

In [20]:
collection1.add(
    documents=df_qa['combined_text'].tolist(),
    metadatas = df_qa.to_dict(orient='records'),
    ids=df_qa.index.astype(str).tolist()
)

In [21]:
collection2 = client.get_or_create_collection(name='medical_device_manual')

In [22]:
df_md = df_md.fillna("")

# Convert everything to string (safe fix)
df_md = df_md.astype(str)

In [23]:
collection2.add(
    documents=df_md['combined_text'].tolist(),
    metadatas=df_md.to_dict(orient="records"),
    ids=df_md.index.astype(str).tolist(),
)

In [24]:
query = "what are the devices relevant to surgery"
results = collection2.query(query_texts=[query], n_results=3)
print(results)

{'ids': [['403', '142', '82']], 'embeddings': None, 'documents': [['Device Name: Orthopedic Implant. Model: Model 4592. Manufacturer: Intuitive Surgical. Indications: Used for post-operative thermal therapy management in hospital recovery units.. Contraindications: Avoid use if patient has coagulopathy or is receiving iodine compounds.', 'Device Name: Surgical Robot. Model: BIO146. Manufacturer: BioMérieux. Indications: Used for therapeutic drug delivery in neonatal patients requiring wound debridement.. Contraindications: Avoid use in patients with mesh implants or mesh implants. Not suitable for infectious disease isolation.', 'Device Name: Anesthesia Machine. Model: C-1766. Manufacturer: Edwards Lifesciences. Indications: Used for intraoperative sterilization monitoring during complex oncological surgeries.. Contraindications: Avoid use in patients with active hepatic dysfunction or compromised respiratory function.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances